# WebGazer experiment analysis with pyxations

This notebook replaces the custom pipeline with **pyxations** tools.

Covers both experiments:
- `antisaccade_experiment/`
- `precision_experiment/`

What pyxations handles:
- Parses WebGazer CSVs (embedded JSON → samples DataFrame)
- Runs REMoDNaV (fixation/saccade detection) as a Python API
- Saves everything in feather format
- Propagates experiment metadata into samples via `behavioral_columns`

What remains as custom code:
- Antisaccade error classification (threshold crossing logic)
- Normalisation relative to baseline
- Precision metrics specific to the experiment

## 0. Setup

In [ ]:
# Install pyxations (uncomment on first run):
# !pip install pyxations==0.3.0

from pathlib import Path
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.feather as feather
import pyxations as pyx

print('pyxations', pyx.__version__ if hasattr(pyx, '__version__') else 'OK')

REPO_ROOT = Path('.').resolve()
print(f'Working dir: {REPO_ROOT}')

---
## 1. Antisaccade experiment

### 1.1 Prepare files for pyxations

pyxations expects `{subject_id}_{...}.csv` (subject first).  
Original files are `antisacadas_{subject_id}.csv` — copies are made in a staging directory, originals are not modified.

In [ ]:
RAW_ANTI     = REPO_ROOT / 'antisaccade_experiment' / 'raw_data'
ANTI_STAGING = REPO_ROOT / 'antisaccade_experiment' / '_staging'
ANTI_STAGING.mkdir(exist_ok=True)

for csv_file in RAW_ANTI.glob('antisacadas_*.csv'):
    subject_id = csv_file.stem.split('_')[1]       # '100', '101', ...
    new_name   = f'{subject_id}_antisacadas.csv'   # '100_antisacadas.csv'
    dest = ANTI_STAGING / new_name
    if not dest.exists():
        shutil.copy(csv_file, dest)

print(f'Files prepared: {len(list(ANTI_STAGING.glob("*.csv")))} subjects')
print([f.name for f in sorted(ANTI_STAGING.glob('*.csv'))[:5]])

### 1.2 Convert to BIDS

In [ ]:
ANTI_BIDS = REPO_ROOT / 'antisaccade_experiment' / 'bids'

bids_path = pyx.dataset_to_bids(
    target_folder_path=ANTI_BIDS,
    files_folder_path=ANTI_STAGING,
    dataset_name='antisaccade',
    session_substrings=1,
    format_name='webgazer',
)

print(f'BIDS created at: {bids_path}')
participants = pd.read_csv(bids_path / 'participants.tsv', sep='\t')
print(f'\nParticipants ({len(participants)}):')
print(participants.head())

### 1.3 Compute derivatives (fixation and saccade detection)

Replaces the `os.system("remodnav ...")` loop that generated ~1000 TSVs.  
pyxations runs REMoDNaV in parallel and saves feather files.  
`behavioral_columns` joins experiment metadata from the source CSV into `samples.feather` by `trial_index` — no separate CSV read is needed downstream.

In [ ]:
pyx.compute_derivatives_for_dataset(
    bids_dataset_folder=bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    num_processes=4,
    behavioral_columns=[
        'typeOfSaccade', 'cueShownAtLeft', 'intraEnd',
        'isTutorial', 'isSaccadeExperiment',
    ],
    overwrite=True,
)

derivatives_path = Path(str(bids_path) + '_derivatives')
print(f'Derivatives at: {derivatives_path}')

### 1.4 Load samples

`samples.feather` columns: `trial_index`, `X`, `Y`, `tSample`, `t_acum`, + behavioral columns.  
No additional CSV read is needed for the analysis.

In [ ]:
def load_subject_samples(derivatives_path, subject_id):
    sub_path = derivatives_path / f'sub-{subject_id}'
    sessions = [s for s in sub_path.iterdir() if s.name.startswith('ses-')]
    return feather.read_feather(sessions[0] / 'samples.feather').reset_index(drop=True)


participants = pd.read_csv(bids_path / 'participants.tsv', sep='\t', dtype=str)
id_map = dict(zip(participants['subject_id'], participants['old_subject_id']))

sub_samples = load_subject_samples(derivatives_path, '0001')
print('Columns in samples.feather:')
print([c for c in sub_samples.columns
       if c not in ['X', 'Y', 'tSample', 't_acum', 'line_number', 'time_elapsed']])

### 1.5 Antisaccade analysis

Trial metadata (`typeOfSaccade`, `cueShownAtLeft`, `intraEnd`) is already in `df_samples` — no CSV needed.  
Detection logic: normalise x by baseline, check threshold crossing.

In [ ]:
def analyze_antisaccade_subject(
    df_samples,
    sacc_type,
    baseline_start=-200.0,
    baseline_end=100.0,
    FILTER=1.5,
    threshold=0.5,
    savgol_flag=False,
):
    """
    Antisaccade analysis using pyxations samples.

    Trial metadata (typeOfSaccade, cueShownAtLeft, intraEnd) is already in
    df_samples thanks to behavioral_columns. No additional CSV read is needed.
    """
    from scipy.signal import savgol_filter

    meta_cols = ['trial_index', 'typeOfSaccade', 'cueShownAtLeft', 'intraEnd',
                 'isTutorial', 'isSaccadeExperiment']
    available  = [c for c in meta_cols if c in df_samples.columns]
    trial_meta = df_samples[available].drop_duplicates('trial_index')

    mask = trial_meta['typeOfSaccade'] == sacc_type
    if 'isTutorial' in trial_meta.columns:
        mask &= trial_meta['isTutorial'] == False
    if 'isSaccadeExperiment' in trial_meta.columns:
        mask &= trial_meta['isSaccadeExperiment'].notna()
    sub_trials = trial_meta[mask]

    errors, n_rejected = 0, 0
    rt_errors, rt_correct, ts_xs_ys = [], [], []

    for _, trial in sub_trials.iterrows():
        trial_idx = trial['trial_index']
        intra_end = trial['intraEnd']
        cue_left  = bool(trial['cueShownAtLeft'])

        t_samps = df_samples[df_samples['trial_index'] == trial_idx]
        if t_samps.empty:
            n_rejected += 1
            continue

        xs = t_samps['X'].to_numpy(dtype=float)
        ys = t_samps['Y'].to_numpy(dtype=float)
        ts = t_samps['tSample'].to_numpy(dtype=float) - intra_end

        mask_base = (ts > baseline_start) & (ts < baseline_end)
        mask_ref  = (ts > 500) & (ts <= 700)
        if not mask_base.any() or not mask_ref.any():
            n_rejected += 1
            continue

        x_base = np.median(xs[mask_base])
        x_ref  = np.median(xs[mask_ref])
        y_base = np.median(ys[mask_base])
        y_ref  = np.median(ys[mask_ref])

        denom_x = np.abs(x_base - x_ref)
        denom_y = np.abs(y_base - y_ref)
        if denom_x == 0 or denom_y == 0:
            n_rejected += 1
            continue

        xs = (xs - x_base) / denom_x
        ys = (ys - y_base) / denom_y

        if cue_left:
            xs = xs * -1

        if np.any(xs > FILTER) or np.any(xs < -FILTER):
            n_rejected += 1
            continue

        if savgol_flag:
            xs = savgol_filter(xs, 5, 2)

        xs_after = xs[ts > baseline_end]
        ts_after = ts[ts > baseline_end]

        if sacc_type == 'prosaccade':
            if np.any(xs_after < -threshold):
                errors += 1
                rt_errors.append(ts_after[np.where(xs_after < -threshold)[0][0]])
            else:
                idxs = np.where(xs_after >= threshold)[0]
                if len(idxs): rt_correct.append(ts_after[idxs[0]])
        else:
            if np.any(xs_after > threshold):
                errors += 1
                rt_errors.append(ts_after[np.where(xs_after > threshold)[0][0]])
            else:
                idxs = np.where(xs_after <= -threshold)[0]
                if len(idxs): rt_correct.append(ts_after[idxs[0]])

        ts_xs_ys.append((ts, xs, ys))

    n_valid = len(sub_trials) - n_rejected
    return {
        'n_total':           len(sub_trials),
        'n_rejected':        n_rejected,
        'n_valid':           n_valid,
        'error_rate':        errors / n_valid if n_valid else np.nan,
        'rt_errors':         rt_errors,
        'rt_error_median':   float(np.median(rt_errors)) if rt_errors else np.nan,
        'rt_correct':        rt_correct,
        'rt_correct_median': float(np.median(rt_correct)) if rt_correct else np.nan,
        'ts_xs_ys':          ts_xs_ys,
    }


# Example: subject 0001
sub_samples = load_subject_samples(derivatives_path, '0001')
for stype in ['prosaccade', 'antisaccade']:
    res = analyze_antisaccade_subject(sub_samples, stype)
    print(f'{stype}: n_valid={res["n_valid"]} | error_rate={res["error_rate"]:.3f} '
          f'| rt_correct={res["rt_correct_median"]:.0f}ms | rt_error={res["rt_error_median"]:.0f}ms')

### 1.6 Loop over all subjects

In [ ]:
all_results = []

for subject_id, old_id in id_map.items():
    try:
        sub_samples = load_subject_samples(derivatives_path, subject_id)
        for stype in ['prosaccade', 'antisaccade']:
            res = analyze_antisaccade_subject(sub_samples, stype)
            all_results.append({
                'subject':           old_id,
                'type':              stype,
                'n_valid':           res['n_valid'],
                'n_rejected':        res['n_rejected'],
                'error_rate':        res['error_rate'],
                'rt_correct_median': res['rt_correct_median'],
                'rt_error_median':   res['rt_error_median'],
            })
    except Exception as e:
        print(f'[{old_id}] Error: {e}')

df_all = pd.DataFrame(all_results)
print(f'Subjects processed: {df_all["subject"].nunique()}')
print()
print(df_all.groupby('type')[['n_valid', 'error_rate', 'rt_correct_median']].mean().round(2))

### 1.7 Per-subject plot

Normalised gaze signals + reaction time KDE.

In [ ]:
def plot_one_subject(ts_xs_ys, rt_errors, rt_correct, sacc_type, subject_id):
    fig, axs = plt.subplots(3, 1, height_ratios=[1, 4, 1],
                            sharex=True, constrained_layout=True)

    for ts, xs, ys in ts_xs_ys:
        axs[1].plot(ts, xs, alpha=0.4, linewidth=0.8)

    axs[1].axhline(y=0.5,  color='k', linestyle='-')
    axs[1].axhline(y=-0.5, color='k', linestyle='-')
    axs[1].axvline(x=0,    color='k', linestyle='-')
    axs[1].set_ylabel('x coordinate (normalized)')
    axs[1].set_xticks(np.arange(-200, 1000, step=100))
    axs[1].set_xlim(-200, 700)
    axs[1].set_ylim(-2, 2)

    if sacc_type == 'prosaccade':
        if rt_errors:  sns.kdeplot(rt_errors, ax=axs[2]);  sns.rugplot(rt_errors, ax=axs[2], height=0.1)
        if rt_correct: sns.kdeplot(rt_correct, ax=axs[0]); sns.rugplot(rt_correct, ax=axs[0], height=0.1)
        axs[2].invert_yaxis()
    else:
        if rt_errors:  sns.kdeplot(rt_errors, ax=axs[0]);  sns.rugplot(rt_errors, ax=axs[0], height=0.1)
        if rt_correct: sns.kdeplot(rt_correct, ax=axs[2]); sns.rugplot(rt_correct, ax=axs[2], height=0.1)
        axs[2].invert_yaxis()

    for ax in [axs[0], axs[2]]:
        ax.set_xlim(-200, 700)

    plt.suptitle(f'{sacc_type} — subject {subject_id}')
    fig.supxlabel('time (ms)')
    plt.show()

In [ ]:
# Change SUBJECT_ID to view a different subject
SUBJECT_ID = '0001'
old_id = id_map[SUBJECT_ID]

sub_samples = load_subject_samples(derivatives_path, SUBJECT_ID)
for stype in ['prosaccade', 'antisaccade']:
    res = analyze_antisaccade_subject(sub_samples, stype)
    plot_one_subject(res['ts_xs_ys'], res['rt_errors'], res['rt_correct'], stype, old_id)

### 1.8 Group summary

In [ ]:
df_all.groupby('type')[['n_valid', 'error_rate', 'rt_correct_median', 'rt_error_median']].mean().round(3)

### 1.9 Group boxplots (error rate and RT)

In [ ]:
from scipy.stats import ranksums

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, label in zip(axes,
                          ['error_rate', 'rt_correct_median'],
                          ['Error rate', 'Correct RT median (ms)']):
    sns.boxplot(data=df_all, x='type', y=col, width=0.3, ax=ax)
    for patch in ax.patches: patch.set_facecolor('w')
    p = ranksums(
        df_all[df_all['type'] == 'prosaccade'][col].dropna(),
        df_all[df_all['type'] == 'antisaccade'][col].dropna(),
    )[1]
    ax.set_title(f'{label} (p={p:.4f})')
    ax.set_ylabel(label)

plt.tight_layout()
plt.show()

print(df_all.groupby('type')[['error_rate', 'rt_correct_median', 'rt_error_median']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, label in zip(axes,
                          ['error_rate', 'rt_correct_median'],
                          ['Error rate', 'Correct RT median (ms)']):
    for stype, grp in df_all.groupby('type'):
        ax.hist(grp[col].dropna(), bins=8, alpha=0.6, label=stype)
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

---
## 2. Precision experiment

### 2.1 Prepare files for pyxations

Files already follow `{subject}_{session}.csv` format.

In [ ]:
RAW_PREC     = REPO_ROOT / 'precision_experiment' / 'raw_data'
PREC_STAGING = REPO_ROOT / 'precision_experiment' / '_staging'
PREC_STAGING.mkdir(exist_ok=True)

raw_files = sorted(RAW_PREC.glob('*.csv'))
print(f'Files found: {len(raw_files)}')
print([f.name for f in raw_files])

In [ ]:
for csv_file in raw_files:
    # Skip pre-processed result files that lack webgazer_data
    header = pd.read_csv(csv_file, nrows=0)
    if "webgazer_data" not in header.columns:
        continue
    dest = PREC_STAGING / csv_file.name
    shutil.copy(csv_file, dest)

print(f'Files in staging: {len(list(PREC_STAGING.glob("*.csv")))}')
print([f.name for f in sorted(PREC_STAGING.glob("*.csv"))])

### 2.2 Convert to BIDS and compute derivatives

In [ ]:
PREC_BIDS = REPO_ROOT / 'precision_experiment' / 'bids'

prec_bids_path = pyx.dataset_to_bids(
    target_folder_path=PREC_BIDS,
    files_folder_path=PREC_STAGING,
    dataset_name='precision',
    session_substrings=2,
    format_name='webgazer',
)

pyx.compute_derivatives_for_dataset(
    bids_dataset_folder=prec_bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    num_processes=4,
    behavioral_columns=['trial-tag', 'center_x', 'center_y', 'start-x', 'start-y'],
    overwrite=True,
)

prec_derivatives_path = Path(str(prec_bids_path) + '_derivatives')
print(f'Derivatives at: {prec_derivatives_path}')

In [ ]:
participants  = pd.read_csv(prec_bids_path / 'participants.tsv', sep='\t', dtype=str)
prec_id_map   = dict(zip(participants['subject_id'], participants['old_subject_id']))
print('Subjects:', prec_id_map)

### 2.3 Precision analysis

Columns `trial-tag`, `center_x`, `center_y`, `start-x`, `start-y` are already in `samples.feather`.  
No CSV read needed.

In [ ]:
def analyze_precision_subject(samples_path, trial_tag='validation-stimulus',
                               first_sample=500, max_var=75):
    samples     = feather.read_feather(samples_path)
    val_samples = samples[samples['trial-tag'] == trial_tag]
    val_trials  = (
        val_samples[['trial_index', 'center_x', 'center_y', 'start-x', 'start-y']]
        .drop_duplicates('trial_index')
    )

    results = []
    for _, row in val_trials.iterrows():
        tidx        = row['trial_index']
        presented_x = row['center_x'] + row['start-x']
        presented_y = row['center_y'] + row['start-y']

        trial_samps = samples[
            (samples['trial_index'] == tidx) & (samples['tSample'] > first_sample)
        ]
        base = dict(trial_index=tidx, presented_point_x=presented_x,
                    presented_point_y=presented_y)
        if trial_samps.empty:
            results.append({**base, 'n_samples': 0, 'horizontal_error_px': np.nan,
                            'vertical_error_px': np.nan, 'total_error_px': np.nan})
            continue

        xs, ys = trial_samps['X'].values, trial_samps['Y'].values
        if np.std(xs) > max_var or np.std(ys) > max_var:
            results.append({**base, 'n_samples': len(trial_samps), 'horizontal_error_px': np.nan,
                            'vertical_error_px': np.nan, 'total_error_px': np.nan})
            continue

        results.append({**base,
            'n_samples':           len(trial_samps),
            'horizontal_error_px': np.abs(xs - presented_x).mean(),
            'vertical_error_px':   np.abs(ys - presented_y).mean(),
            'total_error_px':      np.sqrt((xs - presented_x)**2 + (ys - presented_y)**2).mean(),
        })

    return pd.DataFrame(results)

In [ ]:
all_prec_results = {}

for subject_id, old_id in prec_id_map.items():
    sub_path = prec_derivatives_path / f'sub-{subject_id}'
    if not sub_path.exists():
        print(f'sub-{subject_id} not found, skipping.')
        continue

    for ses_dir in sorted(sub_path.iterdir()):
        if not ses_dir.name.startswith('ses-'):
            continue
        ses_name     = ses_dir.name[4:]
        samples_path = ses_dir / 'samples.feather'
        if not samples_path.exists():
            print(f'  No samples for sub-{subject_id} ses-{ses_name}, skipping.')
            continue

        print(f'Processing sub-{subject_id} ({old_id}), ses-{ses_name}...')
        df_res = analyze_precision_subject(samples_path)
        df_res['subject_id'] = subject_id
        df_res['old_id']     = old_id
        df_res['session']    = ses_name

        key = f'{old_id}_{ses_name}'
        all_prec_results[key] = df_res

        n_valid = df_res['horizontal_error_px'].notna().sum()
        print(f'  Valid trials: {n_valid}/{len(df_res)}')
        print(f'  H error: {df_res["horizontal_error_px"].mean():.1f} px  '
              f'V error: {df_res["vertical_error_px"].mean():.1f} px  '
              f'Total: {df_res["total_error_px"].mean():.1f} px\n')

### 2.4 Error heatmap per validation point

In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms


def confidence_ellipse(x, y, ax, n_std=3.0, facecolor='none', **kwargs):
    cov     = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    ellipse = Ellipse((0, 0),
                      width=np.sqrt(1 + pearson) * 2,
                      height=np.sqrt(1 - pearson) * 2,
                      facecolor=facecolor, **kwargs)
    transf = (transforms.Affine2D()
              .rotate_deg(45)
              .scale(np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std)
              .translate(np.mean(x), np.mean(y)))
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)


def plot_precision_heatmaps(all_prec_results, vmin=0, vmax=300, cmap='Reds'):
    keys = list(all_prec_results.keys())
    n    = len(keys)
    fig, axes = plt.subplots(n, 2, figsize=(5, 2.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row_idx, key in enumerate(keys):
        df_res    = all_prec_results[key]
        unique_xs = sorted(df_res['presented_point_x'].unique())
        unique_ys = sorted(df_res['presented_point_y'].unique(), reverse=True)

        for col_idx, (error_col, label) in enumerate([
            ('horizontal_error_px', 'Horizontal'),
            ('vertical_error_px',   'Vertical'),
        ]):
            grid = np.full((len(unique_ys), len(unique_xs)), np.nan)
            for i, py in enumerate(unique_ys):
                for j, px in enumerate(unique_xs):
                    m = (df_res['presented_point_x'] == px) & (df_res['presented_point_y'] == py)
                    grid[i, j] = df_res.loc[m, error_col].mean()

            ax = axes[row_idx, col_idx]
            im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            ax.set_title(f'{key}\n{label} (px)', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            plt.colorbar(im, ax=ax, shrink=0.8)

    plt.suptitle('Precision error per validation point', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_precision_heatmaps(all_prec_results)

### 2.5 Error over the experiment

In [ ]:
from scipy.stats import linregress


def plot_precision_over_time(all_prec_results):
    palette = sns.color_palette('flare', 8)

    for key, df_res in all_prec_results.items():
        df_plot = df_res.dropna(subset=['horizontal_error_px']).copy().reset_index(drop=True)
        df_plot['abs']   = df_plot.index + 1
        df_plot['block'] = ((df_plot['trial_index'] - 1) % 8 + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        for ax, (error_col, ylabel) in zip(axes, [
            ('horizontal_error_px', 'Horizontal error (px)'),
            ('vertical_error_px',   'Vertical error (px)'),
            ('total_error_px',      'Total error (px)'),
        ]):
            for block_id in sorted(df_plot['block'].unique()):
                sub   = df_plot[df_plot['block'] == block_id]
                color = palette[(int(block_id) - 1) % len(palette)]
                ax.scatter(sub['abs'], sub[error_col], c=[color], alpha=0.4, s=8)

            slope, intercept, *_ = linregress(df_plot['abs'], df_plot[error_col])
            x_range = np.array([df_plot['abs'].min(), df_plot['abs'].max()])
            ax.plot(x_range, slope * x_range + intercept, 'k-', linewidth=2)
            ax.vlines([i * 72 for i in range(1, 9)], 0, 600, colors='k',
                      linewidths=0.5, linestyles='dashed')
            ax.set_xlim(0, len(df_plot) + 5)
            ax.set_ylim(0, 600)
            ax.set_xlabel('Trial')
            ax.set_ylabel(ylabel)

        plt.suptitle(f'Precision error — {key}', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()

        h = df_res['horizontal_error_px'].mean()
        v = df_res['vertical_error_px'].mean()
        t = df_res['total_error_px'].mean()
        print(f'{key}: H={h:.1f} px  V={v:.1f} px  Total={t:.1f} px')


plot_precision_over_time(all_prec_results)

### 2.6 Gaze scatter vs presented point

In [ ]:
def plot_gaze_scatter(all_prec_results, screen_res=(1920, 1080)):
    for key, df_res in all_prec_results.items():
        subject_id = df_res['subject_id'].iloc[0]
        session    = df_res['session'].iloc[0]

        samples = feather.read_feather(
            prec_derivatives_path / f'sub-{subject_id}' / f'ses-{session}' / 'samples.feather'
        )
        val = samples[samples['trial-tag'] == 'validation-stimulus'].copy()

        unique_pts = (
            val[['start-x', 'start-y', 'center_x', 'center_y']]
            .drop_duplicates(['start-x', 'start-y'])
            .reset_index(drop=True)
        )
        unique_pts['px'] = unique_pts['start-x'] + unique_pts['center_x']
        unique_pts['py'] = unique_pts['start-y'] + unique_pts['center_y']

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.set_xlim(0, screen_res[0])
        ax.set_ylim(screen_res[1], 0)
        ax.axvline(screen_res[0] / 2, color='gray', linewidth=0.5, linestyle='--')
        ax.axhline(screen_res[1] / 2, color='gray', linewidth=0.5, linestyle='--')

        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_pts)))
        for color, (_, pt) in zip(colors, unique_pts.iterrows()):
            pt_samps = val[(val['start-x'] == pt['start-x']) & (val['start-y'] == pt['start-y'])]
            xs, ys   = pt_samps['X'].values, pt_samps['Y'].values
            ax.scatter(xs, ys, s=3, alpha=0.2, color=color)
            ax.scatter(pt['px'], pt['py'], marker='+', s=200, linewidths=2, color=color,
                       label=f"({pt['px']:.0f}, {pt['py']:.0f})")
            if len(xs) > 2:
                confidence_ellipse(xs, ys, ax, n_std=2, edgecolor=color, linewidth=1.5)

        ax.set_title(f'{key} — gaze vs presented point', fontsize=10)
        ax.set_xlabel('X (px)')
        ax.set_ylabel('Y (px)')
        ax.legend(fontsize=7, loc='upper right')
        plt.tight_layout()
        plt.show()


plot_gaze_scatter(all_prec_results)